[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egozi/ai-rag-agents-course/blob/main/meeting6-7/transformer_architectures.ipynb)

**Open this notebook in Colab** — click the badge above to launch the notebook on Google Colab (GitHub->Colab).

*If you moved the notebook or use a different branch, edit the link to point to your GitHub repo/branch/path.*

# Transformer Architecture Use Cases

The Transformer is not one architecture — it's a **family** of architectures.
Different tasks call for different variants. This notebook demonstrates each variant
with concrete, runnable examples using local HuggingFace models.

| Architecture | Examples | Attention | Pre-training | Best for |
|---|---|---|---|---|
| **Encoder-only** | BERT, RoBERTa, DistilBERT | Bidirectional | MLM | Classification, NER, embeddings |
| **Decoder-only** | GPT-2, LLaMA, Mistral | Causal (left→right) | Next-token prediction | Text generation, reasoning |
| **Encoder-Decoder** | T5, BART, mBART | Encoder: bi; Decoder: causal | Span corruption / denoising | Summarization, translation, QA |

**What you'll run:**
- Part 1 — Encoder-only: sentiment, zero-shot classification, NER, semantic similarity
- Part 2 — Decoder-only: text generation with GPT-2 *(optional: GPT-4 via API)*
- Part 3 — Encoder-Decoder: summarization, translation, generative QA with T5/BART
- Part 4 — Choosing the right architecture for your task

## Setup

In [ ]:
!pip install -q transformers torch sentence-transformers python-dotenv openai

In [1]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import os

load_dotenv('/home/amir/source/.env')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cpu
PyTorch: 2.5.1+cu124


---
## Part 1 — Encoder-Only (BERT family)

Encoder-only models read the **entire input at once** — every token attends to every other token
(bidirectional attention). This gives rich, context-aware representations ideal for *understanding* tasks.

They **cannot generate** text freely, but they excel at:
- Classifying text (sentiment, topic, intent)
- Labeling tokens (NER, POS tagging)
- Producing embeddings for semantic search and similarity

```
Input: [CLS] The movie was great [SEP]
          ↕     ↕    ↕    ↕    ↕      ← every token sees every other token
       [h0]  [h1] [h2] [h3] [h4]
         ↓
    [CLS] head → classification label
```

### 1.1 Sentiment Analysis

**Model:** `distilbert-base-uncased-finetuned-sst-2-english` (~268 MB)  
A DistilBERT encoder fine-tuned on the Stanford Sentiment Treebank.
The `[CLS]` token representation is fed into a linear classifier.

In [3]:
sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if device == "cuda" else -1
)

# More challenging examples: negation, sarcasm, mixed sentiment, emojis, idioms, long/multi-clause
reviews = [
    "This movie was absolutely fantastic — the acting was superb!",
    "Boring and predictable. I almost fell asleep.",
    "Not bad at all — actually better than I expected.",
    "I loved the visuals, but the plot? Not so much.",
    "What a masterpiece! ...said no one ever.",
    "I laughed, I cried, I am still confused. Mixed feelings.",
    "The characters were great; pacing, however, killed it.",
    "An utter waste of 2 hours. Couldn't be worse.",
    "Nice soundtrack, weak story. 6/10.",
    "Absolutely loved it 😀❤️ — would watch again!",
    "Terrible. Brilliant? Depends on your taste — ambiguous review.",
    "I fell asleep halfway but woke up for the finale which was okay.",
    "None of the above: it's fine, I guess. Neutral overall.",
]

results = sentiment(reviews)
for i, (text, result) in enumerate(zip(reviews, results), start=1):
    label = result['label']
    score = result['score']
    bar = '█' * int(score * 20)
    print(f"{i:2d}. [{label:8s}] {score:.2f} {bar}")
    print(f"    → {text}")
    print()

 1. [POSITIVE] 1.00 ███████████████████
    → This movie was absolutely fantastic — the acting was superb!

 2. [NEGATIVE] 1.00 ███████████████████
    → Boring and predictable. I almost fell asleep.

 3. [POSITIVE] 1.00 ███████████████████
    → Not bad at all — actually better than I expected.

 4. [NEGATIVE] 1.00 ███████████████████
    → I loved the visuals, but the plot? Not so much.

 5. [POSITIVE] 0.83 ████████████████
    → What a masterpiece! ...said no one ever.

 6. [NEGATIVE] 0.87 █████████████████
    → I laughed, I cried, I am still confused. Mixed feelings.

 7. [POSITIVE] 0.83 ████████████████
    → The characters were great; pacing, however, killed it.

 8. [NEGATIVE] 1.00 ███████████████████
    → An utter waste of 2 hours. Couldn't be worse.

 9. [POSITIVE] 0.99 ███████████████████
    → Nice soundtrack, weak story. 6/10.

10. [POSITIVE] 1.00 ███████████████████
    → Absolutely loved it 😀❤️ — would watch again!

11. [NEGATIVE] 0.79 ███████████████
    → Terrible. Br

In [20]:
# Manual (non-pipeline) sentiment analysis using tokenizer + model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# model.config


In [17]:
tokens = tokenizer(reviews[0])
# print(tokenizer.convert_ids_to_tokens(tokens['input_ids']))

In [9]:
# Tokenize in batch and move tensors to the chosen device
inputs = tokenizer(reviews, return_tensors='pt', truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

inputs

{'input_ids': tensor([[  101,  2023,  3185,  2001,  7078, 10392,  1517,  1996,  3772,  2001,
          21688,   999,   102,     0,     0,     0,     0,     0],
         [  101, 11771,  1998, 21425,  1012,  1045,  2471,  3062,  6680,  1012,
            102,     0,     0,     0,     0,     0,     0,     0],
         [  101,  2025,  2919,  2012,  2035,  1517,  2941,  2488,  2084,  1045,
           3517,  1012,   102,     0,     0,     0,     0,     0],
         [  101,  1045,  3866,  1996, 26749,  1010,  2021,  1996,  5436,  1029,
           2025,  2061,  2172,  1012,   102,     0,     0,     0],
         [  101,  2054,  1037, 17743,   999,  1012,  1012,  1012,  2056,  2053,
           2028,  2412,  1012,   102,     0,     0,     0,     0],
         [  101,  1045,  4191,  1010,  1045,  6639,  1010,  1045,  2572,  2145,
           5457,  1012,  3816,  5346,  1012,   102,     0,     0],
         [  101,  1996,  3494,  2020,  2307,  1025, 15732,  1010,  2174,  1010,
           2730,  2009,  

In [18]:
# Forward pass (no grad) and softmax to probabilities
with torch.no_grad():
    outputs = model(**inputs)

outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-4.3095,  4.6771],
        [ 4.5430, -3.6989],
        [-3.6040,  3.9086],
        [ 4.0534, -3.3141],
        [-0.6292,  0.9247],
        [ 1.0747, -0.8672],
        [-0.7016,  0.8735],
        [ 4.5829, -3.7009],
        [-2.2727,  2.4722],
        [-4.3003,  4.6183],
        [ 0.7783, -0.5254],
        [-3.4466,  3.6543],
        [ 1.4741, -1.0787]]), hidden_states=None, attentions=None)

In [ ]:
logits = outputs.logits
probs = F.softmax(logits, dim=-1).cpu().numpy()
id2label = model.config.id2label  # mapping from index -> label string

In [ ]:
for i, (text, prob) in enumerate(zip(reviews, probs), start=1):
    label_idx = int(prob.argmax())
    label = id2label[label_idx]
    score = float(prob[label_idx])
    bar = '█' * int(score * 20)
    print(f"{i:2d}. [{label:8s}] {score:.2f} {bar}")
    print(f"    → {text}")
    print()

### 1.2 Zero-Shot Classification

**Model:** `facebook/bart-large-mnli` (~1.6 GB) — an encoder-decoder fine-tuned on Natural Language Inference (NLI).  
Given a text and a set of candidate labels, the model determines if the text *entails* each label.

> This is powerful because you can classify into **any category without retraining**.

In [ ]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == "cuda" else -1
)

texts = [
    "The central bank raised interest rates to combat inflation.",
    "Scientists discovered a new species of deep-sea fish near the Mariana Trench.",
    "The team scored three goals in the second half to secure the championship.",
]

candidate_labels = ["finance & economics", "science & nature", "sports", "politics", "technology"]

for text in texts:
    result = zero_shot(text, candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"Text: {text[:70]}")
    print(f"  → {top_label} ({top_score:.2%})")
    # Show top 3
    for label, score in zip(result['labels'][:3], result['scores'][:3]):
        bar = '▓' * int(score * 30)
        print(f"     {label:25s} {score:.2%} {bar}")
    print()

In [23]:
# Manual (non-pipeline) zero-shot classification using NLI (facebook/bart-large-mnli)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

model_name_nli = 'facebook/bart-large-mnli'
tokenizer_nli = AutoTokenizer.from_pretrained(model_name_nli)
model_nli = AutoModelForSequenceClassification.from_pretrained(model_name_nli).to(device)

In [27]:
pairs = [(texts[0], f'This example is about {label}.') for label in candidate_labels]
premises = [p[0] for p in pairs]
hypotheses = [p[1] for p in pairs]
enc = tokenizer_nli(premises, hypotheses, return_tensors='pt', truncation=True, padding=True)
enc = {k: v.to(device) for k, v in enc.items()}

premises

['The central bank raised interest rates to combat inflation.',
 'The central bank raised interest rates to combat inflation.',
 'The central bank raised interest rates to combat inflation.',
 'The central bank raised interest rates to combat inflation.',
 'The central bank raised interest rates to combat inflation.']

In [ ]:
# For each text, score each candidate label by NLI entailment probability
for text in texts:
    pairs = [(text, f'This example is about {label}.') for label in candidate_labels]
    premises = [p[0] for p in pairs]
    hypotheses = [p[1] for p in pairs]
    enc = tokenizer_nli(premises, hypotheses, return_tensors='pt', truncation=True, padding=True)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model_nli(**enc)
    logits = out.logits  # shape: (num_labels, 3) -- [contradiction, neutral, entailment]
    probs = F.softmax(logits, dim=-1).cpu().numpy()

    # find index corresponding to entailment label in model config
    entail_idx = None
    for k, v in model_nli.config.id2label.items():
        if str(v).lower().startswith('entail'):
            entail_idx = int(k)
            break
    if entail_idx is None:
        entail_idx = 2  # fallback (common mapping: 0=CONTRADICTION,1=NEUTRAL,2=ENTAILMENT)

    scores = [(label, float(probs[i, entail_idx])) for i, label in enumerate(candidate_labels)]
    scores_sorted = sorted(scores, key=lambda x: x[1], reverse=True)

    top_label, top_score = scores_sorted[0]
    print(f"Text: {text[:80]}")
    print(f"  → Predicted: {top_label} ({top_score:.2%})")
    for label, score in scores_sorted[:5]:
        bar = '▓' * int(score * 30)
        print(f"     {label:25s} {score:.2%} {bar}")
    print()

### 1.3 Named Entity Recognition (NER)

**Model:** `dslim/bert-base-NER` (~416 MB) — BERT fine-tuned on CoNLL-2003.  
Each token is classified into an entity type: PER (person), ORG (organisation), LOC (location), MISC.

Unlike sentence-level classification, NER is a **token-level** task — every token gets its own label.

In [28]:
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",   # merge sub-word tokens into whole entities
    device=0 if device == "cuda" else -1
)

sentences = [
    "Elon Musk founded SpaceX in Hawthorne, California in 2002.",
    "The European Central Bank, headquartered in Frankfurt, set new rates last Thursday.",
    "Marie Curie was born in Warsaw, Poland and later worked at the University of Paris.",
]

for sentence in sentences:
    entities = ner(sentence)
    print(f"Text: {sentence}")
    print(f"  Entities found: {len(entities)}")
    for ent in entities:
        print(f"    [{ent['entity_group']:5s}] '{ent['word']}' (score: {ent['score']:.2f})")
    print()

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Text: Elon Musk founded SpaceX in Hawthorne, California in 2002.
  Entities found: 5
    [ORG  ] 'Elon' (score: 0.60)
    [PER  ] 'Musk' (score: 0.72)
    [ORG  ] 'SpaceX' (score: 1.00)
    [LOC  ] 'Hawthorne' (score: 1.00)
    [LOC  ] 'California' (score: 1.00)

Text: The European Central Bank, headquartered in Frankfurt, set new rates last Thursday.
  Entities found: 2
    [ORG  ] 'European Central Bank' (score: 1.00)
    [LOC  ] 'Frankfurt' (score: 1.00)

Text: Marie Curie was born in Warsaw, Poland and later worked at the University of Paris.
  Entities found: 4
    [PER  ] 'Marie Curie' (score: 0.99)
    [LOC  ] 'Warsaw' (score: 1.00)
    [LOC  ] 'Poland' (score: 1.00)
    [ORG  ] 'University of Paris' (score: 1.00)



### 1.4 Semantic Embeddings & Similarity

**Model:** `sentence-transformers/all-MiniLM-L6-v2` (~90 MB) — a tiny but powerful sentence encoder.  
Each sentence is mapped to a 384-dimensional vector. Semantically similar sentences land close together in this space.

This is the foundation of **semantic search** and **RAG retrieval**.

In [ ]:
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "A dog is playing fetch in the park.",       # 0 - animals
    "The puppy chased the ball across the lawn.", # 1 - animals (similar to 0)
    "Neural networks learn hierarchical features.",# 2 - ML
    "Deep learning models extract patterns from data.", # 3 - ML (similar to 2)
    "The stock market dropped sharply today.",    # 4 - finance
]

embeddings = embedder.encode(sentences, normalize_embeddings=True)
print(f"Embedding shape: {embeddings.shape}  ({len(sentences)} sentences × 384 dims)\n")

# Cosine similarity matrix (dot product since embeddings are normalized)
sim_matrix = embeddings @ embeddings.T

# Display as heatmap
short_labels = [s[:35] + "..." for s in sentences]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=40, ha='right', fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)
plt.colorbar(im, ax=ax, label="Cosine Similarity")
ax.set_title("Semantic Similarity Matrix (all-MiniLM-L6-v2)")
for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()

print("Key pairs:")
print(f"  Dog / Puppy (both animals): {sim_matrix[0,1]:.3f}")
print(f"  Neural nets / Deep learning (both ML): {sim_matrix[2,3]:.3f}")
print(f"  Dog / Stock market (unrelated): {sim_matrix[0,4]:.3f}")

#### Semantic Search Demo

In [ ]:
# A small knowledge base
documents = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is pre-trained with masked language modeling on large text corpora.",
    "GPT models generate text autoregressively by predicting the next token.",
    "T5 frames every NLP task as text-to-text and uses an encoder-decoder architecture.",
    "Retrieval-Augmented Generation (RAG) combines a retriever with a generative model.",
    "Embeddings are dense vector representations that capture semantic meaning.",
    "Fine-tuning adapts a pre-trained model to a specific downstream task.",
    "Attention scores determine how much focus each token gives to every other token.",
]

doc_embeddings = embedder.encode(documents, normalize_embeddings=True)

def semantic_search(query: str, k: int = 3) -> list:
    query_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = doc_embeddings @ query_emb
    top_k = np.argsort(scores)[::-1][:k]
    return [(documents[i], float(scores[i])) for i in top_k]

queries = [
    "How does BERT learn representations?",
    "What architecture is used for text generation?",
    "How do you adapt a model to a new domain?",
]

for q in queries:
    print(f"Query: '{q}'")
    for doc, score in semantic_search(q, k=2):
        print(f"  [{score:.3f}] {doc}")
    print()

### Exercise 1 — Encoder: Custom Zero-Shot Topic Router

You are building a customer support bot. Use zero-shot classification to route
incoming support tickets to the right department.

**Your task:**
1. Define candidate department labels (e.g. `billing`, `technical support`, `returns`, `account`, `shipping`)
2. Run zero-shot classification on the sample tickets below
3. Print the ticket, predicted department, and confidence score
4. Mark any prediction below 60% confidence as `"needs human review"`

In [ ]:
tickets = [
    "I was charged twice for my last order. Please refund the duplicate.",
    "My password reset email never arrived and I can't log in.",
    "The laptop I received has a cracked screen. I want to send it back.",
    "Where is my package? It's been 2 weeks since I ordered.",
    "I need an invoice for my corporate purchase from last month.",
]

# TODO: Define your department labels
departments = None  # Your code

# TODO: For each ticket, run zero_shot() and print the result
# Flag low-confidence predictions (< 0.60) with "needs human review"
for ticket in tickets:
    result = None  # Your code
    # Print department and confidence
    pass

#### Solution

In [ ]:
departments = ["billing", "technical support", "returns & refunds", "shipping & delivery", "account management"]

for ticket in tickets:
    result = zero_shot(ticket, departments)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    flag = " ⚠️  needs human review" if top_score < 0.60 else ""
    print(f"Ticket: {ticket[:60]}")
    print(f"  → [{top_label}] {top_score:.1%}{flag}")
    print()

---
## Part 2 — Decoder-Only (GPT family)

Decoder-only models generate text **one token at a time**, left to right.
Each token can only attend to previous tokens (causal masking) — they cannot
look ahead. This makes them ideal for *generation* tasks.

```
Input: The  cat  sat  on   the
         ↓    ↓    ↓    ↓    ↓
       [h0] [h1] [h2] [h3] [h4]   ← each token only sees tokens to its left
                                ↓
                               mat  (predicted next token)
```

**Local model:** `gpt2` (124M parameters, ~548 MB)  
**Optional API:** OpenAI `gpt-4o-mini` for higher quality generation

### 2.1 Text Generation (Local — GPT-2)

In [29]:
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0 if device == "cuda" else -1
)

prompts = [
    "In a future where AI assistants are everywhere,",
    "The key difference between supervised and unsupervised learning is",
    "def fibonacci(n):\n    \"\"\"",
]

for prompt in prompts:
    output = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        num_return_sequences=1,
        pad_token_id=50256   # GPT-2 has no pad token; use EOS
    )
    print(f"Prompt: '{prompt[:50]}'")
    print(f"Generated:\n{output[0]['generated_text']}")
    print("-" * 60)

Prompt: 'In a future where AI assistants are everywhere,'
Generated:
In a future where AI assistants are everywhere, the answer is to move to smarter algorithms and make smart choices.

"This is a revolution, and it's going to be big for the future of human interaction and the world," said Timo. "That's why AI is such a big problem right now."

AI assistants are
------------------------------------------------------------
Prompt: 'The key difference between supervised and unsuperv'
Generated:
The key difference between supervised and unsupervised learning is that supervised learning relies on the ability to predict the outcome of a task with the intent to train it. Unsupervised learning is a learning method that involves learning and applying an algorithm to the data to produce a predetermined result. In order to achieve this, a supervised learning algorithm has to be able to
------------------------------------------------------------
Prompt: 'def fibonacci(n):
    """'
Generated:
def

### 2.2 Effect of Sampling Parameters

The same prompt can produce very different results depending on how you sample.
Let's compare three strategies side-by-side.

In [ ]:
prompt = "Scientists recently discovered that"

strategies = [
    {"label": "Greedy (deterministic)",     "do_sample": False},
    {"label": "Low temperature (focused)",  "do_sample": True, "temperature": 0.3, "top_p": 0.9},
    {"label": "High temperature (creative)","do_sample": True, "temperature": 1.4, "top_p": 0.95},
]

for strat in strategies:
    label = strat.pop("label")
    output = generator(
        prompt,
        max_new_tokens=50,
        pad_token_id=50256,
        **strat
    )
    generated = output[0]['generated_text'][len(prompt):].strip()
    print(f"[{label}]")
    print(f"  {generated}")
    print()

### 2.3 Optional: OpenAI API (GPT-4o-mini)

GPT-2 is small and shows the architecture, but modern decoder models are far more capable.
Run the cell below if you have `OPENAI_API_KEY` set in your `.env` file.

In [ ]:
import openai

if os.getenv("OPENAI_API_KEY"):
    client = openai.OpenAI()

    tasks = [
        "Explain the difference between BERT and GPT in two sentences.",
        "Write a Python function that reverses a linked list.",
        "What is the capital of France? Answer in one word.",
    ]

    for task in tasks:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": task}],
            max_tokens=150,
            temperature=0.7,
        )
        answer = response.choices[0].message.content.strip()
        print(f"Task: {task}")
        print(f"GPT-4o-mini: {answer}")
        print("-" * 60)
else:
    print("OPENAI_API_KEY not found — skipping API demo.")
    print("Set the key in /home/amir/source/.env to enable this section.")

### Exercise 2 — Decoder: Multi-Turn Story Continuation

Decoder models are naturally suited for *continuation* — given context, predict what comes next.

**Your task:**
1. Start with the opening sentence below
2. Generate a continuation (30–50 tokens)
3. Take the last sentence of that continuation as the new prompt
4. Repeat 3 times to build a short 4-part story
5. Print the full story at the end

*(Hint: `output[0]['generated_text']` contains the full text including the prompt)*

In [ ]:
opening = "The last astronaut on the space station opened the airlock and saw something unexpected."

# TODO: iteratively generate 3 more turns, using the end of each output as the new prompt
# Print the assembled story at the end
story = opening

for turn in range(3):
    # Your code: generate continuation from current story tail
    pass

print(story)

#### Solution

In [ ]:
story = opening

for turn in range(3):
    # Use last 200 chars as context (GPT-2 has a 1024 token limit)
    context = story[-200:]
    output = generator(
        context,
        max_new_tokens=45,
        do_sample=True,
        temperature=0.9,
        top_p=0.9,
        pad_token_id=50256
    )
    # Extract only the new tokens (strip the context prefix)
    new_text = output[0]['generated_text'][len(context):].strip()
    story += " " + new_text
    print(f"Turn {turn + 1}: {new_text}")
    print()

print("=" * 60)
print("Full story:")
print(story)

---
## Part 3 — Encoder-Decoder (T5 / BART family)

Encoder-decoder models have **two separate stacks**:
- The **encoder** reads the input bidirectionally (like BERT)
- The **decoder** generates the output autoregressively (like GPT), but it also attends to the encoder output via **cross-attention**

```
Input sentence ──► [Encoder] ──► context vectors
                                       │
                               cross-attention
                                       │
Output prefix ──► [Decoder] ──────────┘──► next output token
```

This architecture is ideal for tasks where the input and output are **different sequences**:
summarization, translation, question answering, style transfer.

### 3.1 Summarization

**Model:** `sshleifer/distilbart-cnn-12-6` (~1.2 GB) — a distilled version of BART fine-tuned on CNN/DailyMail news articles.

T5 alternative (smaller): `t5-small` with prefix `"summarize: "`

In [ ]:
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=0 if device == "cuda" else -1
)

articles = [
    """
    Researchers at MIT have developed a new technique for training large language models
    that significantly reduces the amount of memory required during training. The method,
    called gradient checkpointing with selective recomputation, allows models with billions
    of parameters to be trained on hardware that would otherwise be insufficient. The team
    demonstrated that a 7-billion parameter model could be trained on consumer GPUs with
    24 GB of VRAM, compared to the 80 GB previously required. The technique works by
    discarding intermediate activations during the forward pass and recomputing them during
    backpropagation, trading compute time for memory efficiency. The researchers estimate
    this will make large-scale AI research accessible to far more institutions worldwide.
    """,
    """
    The European Union has announced a comprehensive new regulatory framework for artificial
    intelligence systems, requiring companies deploying high-risk AI in areas such as hiring,
    credit scoring, and critical infrastructure to undergo mandatory third-party audits.
    Companies that fail to comply could face fines of up to 6% of global annual revenue.
    The regulation distinguishes between minimal-risk AI, such as spam filters, and
    high-risk AI that may affect fundamental rights. Member states must designate national
    supervisory authorities within 18 months of the law coming into force.
    """,
]

for i, article in enumerate(articles, 1):
    summary = summarizer(
        article.strip(),
        max_length=80,
        min_length=25,
        do_sample=False
    )
    print(f"Article {i} ({len(article.split())} words):")
    print(f"  Summary: {summary[0]['summary_text']}")
    print()

#### T5 alternative (smaller, ~242 MB)

T5 treats every task as text-to-text: you prefix your input with a task description.

In [ ]:
t5_summarizer = pipeline(
    "summarization",
    model="t5-small",
    tokenizer="t5-small",
    device=0 if device == "cuda" else -1
)

# T5 expects the "summarize: " prefix
text = "summarize: " + articles[0].strip()
result = t5_summarizer(text, max_length=60, min_length=15, do_sample=False)
print(f"T5-small summary: {result[0]['summary_text']}")

### 3.2 Translation

**Model:** `Helsinki-NLP/opus-mt-en-fr` (~298 MB) — a small, fast Marian MT model trained on OPUS parallel corpora.
These models are encoder-decoder architectures specifically optimized for neural machine translation.

In [ ]:
# English → French
en_fr = pipeline(
    "translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr",
    device=0 if device == "cuda" else -1
)

sentences_en = [
    "Transformers have revolutionized natural language processing.",
    "The model was trained on a large corpus of text data.",
    "Attention mechanisms allow models to focus on relevant parts of the input.",
    "Hello, how are you today?",
]

print("English → French translations:\n")
for sent in sentences_en:
    result = en_fr(sent, max_length=100)
    translation = result[0]['translation_text']
    print(f"  EN: {sent}")
    print(f"  FR: {translation}")
    print()

In [ ]:
# T5 can also translate using a text-to-text prefix
t5_translator = pipeline(
    "translation_en_to_fr",
    model="t5-small",
    device=0 if device == "cuda" else -1
)

test = "The weather is beautiful today."
result_helsinki = en_fr(test)[0]['translation_text']
result_t5 = t5_translator(test)[0]['translation_text']

print(f"Source:        {test}")
print(f"Helsinki-NLP:  {result_helsinki}")
print(f"T5-small:      {result_t5}")

### 3.3 Generative Question Answering (Flan-T5)

**Model:** `google/flan-t5-small` (~308 MB) — T5 fine-tuned with instruction-following data.  
Unlike extractive QA (which picks a span from the context), Flan-T5 **generates** the answer.

This is the precursor to modern instruction-tuned models like ChatGPT.

In [ ]:
flan_t5 = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    device=0 if device == "cuda" else -1
)

# Closed-book QA (no context provided — knowledge from pre-training)
questions = [
    "What is the capital of France?",
    "Who wrote the play Romeo and Juliet?",
    "What does BERT stand for?",
    "In one sentence, what is a neural network?",
]

print("Closed-book QA (Flan-T5-small):\n")
for q in questions:
    result = flan_t5(q, max_new_tokens=50)
    answer = result[0]['generated_text']
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

In [ ]:
# Context-grounded QA (provide a passage, ask a question)
context = """
The transformer architecture was introduced in the 2017 paper "Attention Is All You Need"
by Vaswani et al. at Google. It replaced recurrent networks with a purely attention-based
approach, enabling much better parallelization during training. BERT, released by Google
in 2018, uses only the encoder part of the transformer and is pre-trained with masked
language modeling. GPT, released by OpenAI in 2018, uses only the decoder part and is
pre-trained with causal language modeling.
"""

grounded_questions = [
    "What year was the transformer paper published?",
    "What is BERT pre-trained with?",
    "Who released GPT?",
    "What did transformers replace?",
]

print("Context-grounded QA:\n")
for q in grounded_questions:
    prompt = f"Answer the question based on the context.\nContext: {context.strip()}\nQuestion: {q}"
    result = flan_t5(prompt, max_new_tokens=40)
    print(f"Q: {q}")
    print(f"A: {result[0]['generated_text']}")
    print()

### Exercise 3 — Encoder-Decoder: Summarize + Translate Pipeline

Build a two-stage pipeline:
1. **Summarize** a long English article using `distilbart` or `t5-small`
2. **Translate** the summary to French using `Helsinki-NLP/opus-mt-en-fr`
3. Print: original length, summary, and French translation

Test it on the two articles defined earlier in Part 3.1.

In [ ]:
def summarize_and_translate(text: str, max_summary_length: int = 60) -> dict:
    """
    Summarize an English text and translate the summary to French.
    Returns a dict with 'original_words', 'summary', 'translation'
    """
    # TODO: Step 1 — summarize the text
    summary = None  # Your code
    
    # TODO: Step 2 — translate the summary
    translation = None  # Your code
    
    return {
        "original_words": len(text.split()),
        "summary": summary,
        "translation": translation,
    }

# Test on both articles
for i, article in enumerate(articles, 1):
    result = summarize_and_translate(article.strip())
    print(f"Article {i} ({result['original_words']} words)")
    print(f"  Summary (EN): {result['summary']}")
    print(f"  Summary (FR): {result['translation']}")
    print()

#### Solution

In [ ]:
def summarize_and_translate(text: str, max_summary_length: int = 60) -> dict:
    # Step 1: summarize
    summary_result = summarizer(text, max_length=max_summary_length, min_length=15, do_sample=False)
    summary = summary_result[0]['summary_text']
    
    # Step 2: translate
    translation_result = en_fr(summary, max_length=150)
    translation = translation_result[0]['translation_text']
    
    return {
        "original_words": len(text.split()),
        "summary": summary,
        "translation": translation,
    }

for i, article in enumerate(articles, 1):
    result = summarize_and_translate(article.strip())
    print(f"Article {i} ({result['original_words']} words)")
    print(f"  Summary (EN): {result['summary']}")
    print(f"  Summary (FR): {result['translation']}")
    print()

---
## Part 4 — Choosing the Right Architecture

### Decision Guide

| Your task | Recommended architecture | Why |
|---|---|---|
| Classify a document | **Encoder-only** (BERT, DistilBERT) | Rich bidirectional context, lightweight |
| Label each token (NER, POS) | **Encoder-only** | Token-level representations |
| Embed sentences for search | **Encoder-only** (SBERT) | Fixed-size dense vectors |
| Generate free-form text | **Decoder-only** (GPT, LLaMA) | Autoregressive, trained for generation |
| Code completion / reasoning | **Decoder-only** | Strong on long-range causal reasoning |
| Summarize a document | **Encoder-Decoder** (BART, T5) | Reads full input, generates shorter output |
| Translate between languages | **Encoder-Decoder** (Helsinki-NLP, mBART) | Cross-lingual encoder + decoder |
| Instruction following | **Decoder-only** (instruction-tuned) or **Enc-Dec** (Flan-T5) | RLHF / SFT fine-tuning |
| RAG (retrieval + answer) | **Encoder** for retrieval + **Decoder** or **Enc-Dec** for generation | Two-stage pipeline |

> **Rule of thumb:** If the output length ≈ input length → encoder-only.  
> If output is new text → decoder-only or encoder-decoder.  
> If output is a transformation of the input → encoder-decoder.

### Architecture Comparison on the Same Task

Let's ask all three architectures to "answer" the same question and observe the differences.

In [ ]:
question = "What is the main advantage of transformer models over RNNs?"
context = (
    "Transformer models process all tokens in parallel using self-attention, "
    "while RNNs process tokens sequentially. This parallelism makes transformers "
    "much faster to train on modern hardware. Transformers also capture long-range "
    "dependencies more effectively because attention has direct access to any position."
)

print(f"Question: {question}")
print(f"Context: {context[:80]}...")
print()

# --- Approach 1: Encoder-only extractive QA ---
# Extracts a span directly from the context — no generation
extractive_qa = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=0 if device == "cuda" else -1
)
ext_result = extractive_qa(question=question, context=context)
print(f"[Encoder-only] Extractive QA (DistilBERT):")
print(f"  Answer: '{ext_result['answer']}' (score: {ext_result['score']:.3f})")
print(f"  Note: copied directly from context — no new words generated")
print()

# --- Approach 2: Encoder-Decoder generative QA ---
# Generates an answer (may rephrase or go beyond the context)
flan_prompt = f"Answer based on context.\nContext: {context}\nQuestion: {question}"
gen_result = flan_t5(flan_prompt, max_new_tokens=60)
print(f"[Encoder-Decoder] Generative QA (Flan-T5-small):")
print(f"  Answer: {gen_result[0]['generated_text']}")
print(f"  Note: generated text — may rephrase or synthesize")
print()

# --- Approach 3: Decoder-only ---
# Continues the prompt — best with instruction-tuned models
decoder_prompt = f"Context: {context}\nQuestion: {question}\nAnswer:"
dec_result = generator(decoder_prompt, max_new_tokens=60, do_sample=False, pad_token_id=50256)
generated_part = dec_result[0]['generated_text'][len(decoder_prompt):].strip()
print(f"[Decoder-only] GPT-2 completion:")
print(f"  Answer: {generated_part}")
print(f"  Note: GPT-2 is not instruction-tuned, so quality varies")

---
## Summary

| | Encoder-only | Decoder-only | Encoder-Decoder |
|---|---|---|---|
| **Attention** | Bidirectional | Causal | Encoder: bi; Decoder: causal + cross |
| **Pre-training** | MLM (BERT), replaced tokens | Next-token prediction | Span corruption, denoising |
| **Generates new text?** | No | Yes | Yes |
| **Reads full input?** | Yes | Yes (during training) | Yes |
| **Typical size** | 66M–340M | 117M–70B+ | 220M–11B |
| **Best local model** | DistilBERT, all-MiniLM | GPT-2, GPT-J | T5-small, Flan-T5, DistilBART |
| **Best API model** | OpenAI `text-embedding-*` | GPT-4, Claude | — |

### Models used in this notebook

| Model | Size | Task |
|---|---|---|
| `distilbert-base-uncased-finetuned-sst-2-english` | ~268 MB | Sentiment analysis |
| `facebook/bart-large-mnli` | ~1.6 GB | Zero-shot classification |
| `dslim/bert-base-NER` | ~416 MB | Named entity recognition |
| `sentence-transformers/all-MiniLM-L6-v2` | ~90 MB | Semantic embeddings |
| `gpt2` | ~548 MB | Text generation |
| `sshleifer/distilbart-cnn-12-6` | ~1.2 GB | Summarization |
| `t5-small` | ~242 MB | Summarization + translation |
| `Helsinki-NLP/opus-mt-en-fr` | ~298 MB | Translation |
| `google/flan-t5-small` | ~308 MB | Generative QA |
| `distilbert-base-cased-distilled-squad` | ~261 MB | Extractive QA |

### References

- Devlin et al., [BERT: Pre-training of Deep Bidirectional Transformers](https://arxiv.org/abs/1810.04805) (2018)
- Radford et al., [Language Models are Unsupervised Multitask Learners](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (GPT-2, 2019)
- Raffel et al., [Exploring the Limits of Transfer Learning with T5](https://arxiv.org/abs/1910.10683) (2020)
- Lewis et al., [BART: Denoising Sequence-to-Sequence Pre-training](https://arxiv.org/abs/1910.13461) (2019)
- HuggingFace [Pipelines documentation](https://huggingface.co/docs/transformers/main_classes/pipelines)